In [ ]:
# -*- coding: utf-8 -*-
"""
Random Forest para compensação de temperatura
Treino: curvas em {0, 10, 40, 60 °C}
Teste : curvas em {–10, 30, 50, 70 °C}
"""

import re, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ===================== PARÂMETROS =====================
REF_TEMP      = 20
FREQ_MIN_KHZ  = 30
FREQ_MAX_KHZ  = 50
PKL_TREINO    = "base_treino.pkl"
PKL_PROVA     = "base_prova (1).pkl"
TEMPS_TREINO  = {0, 10, 40, 60}
TEMPS_PROVA   = {-10, 30, 50, 70}

# ===================== HELPERS =====================
def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None

def get_freq_columns(df, fmin_khz, fmax_khz):
    cols, freqs = [], []
    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None:
            fk = f/1e3
            if fmin_khz <= fk <= fmax_khz:
                cols.append(c); freqs.append(f)
    order = np.argsort(freqs)
    return [cols[i] for i in order], np.array(freqs, float)[order]

# ===================== LOAD =====================
base_tr = pd.read_pickle(PKL_TREINO)
base_te = pd.read_pickle(PKL_PROVA)

freq_cols_tr, _ = get_freq_columns(base_tr, FREQ_MIN_KHZ, FREQ_MAX_KHZ)
freq_cols_te, _ = get_freq_columns(base_te, FREQ_MIN_KHZ, FREQ_MAX_KHZ)
common_cols = [c for c in freq_cols_tr if c in freq_cols_te]
fhz = np.array([extract_freq_hz(c) for c in common_cols], float)
order = np.argsort(fhz); common_cols = [common_cols[i] for i in order]; fhz = fhz[order]
fkHz = fhz/1e3

# Subconjuntos fixos
tr_restr = base_tr[base_tr["temp_c"].isin(TEMPS_TREINO)].copy()
te_restr = base_te[base_te["temp_c"].isin(TEMPS_PROVA)].copy()

X_tr = tr_restr[common_cols].to_numpy(float)
X_te = te_restr[common_cols].to_numpy(float)
T_tr = tr_restr["temp_c"].to_numpy(float)
T_te = te_restr["temp_c"].to_numpy(float)

# ===================== REFERÊNCIA 20 °C =====================
pool_20 = []
if (base_tr["temp_c"]==REF_TEMP).any():
    pool_20.append(base_tr.loc[base_tr["temp_c"]==REF_TEMP, common_cols].to_numpy(float))
if (base_te["temp_c"]==REF_TEMP).any():
    pool_20.append(base_te.loc[base_te["temp_c"]==REF_TEMP, common_cols].to_numpy(float))
assert len(pool_20)>0, "Não há curva real @20°C!"
y_ref = np.median(np.vstack(pool_20), axis=0)

# ===================== TREINO RANDOM FOREST =====================
# Entrada: curva original + temperatura
X_train = np.hstack([X_tr, T_tr[:,None]])
Y_train = np.tile(y_ref, (len(X_tr),1))   # alvo = curva de referência

# Modelo multissaída
rf = MultiOutputRegressor(RandomForestRegressor(
    n_estimators=300, max_depth=15, n_jobs=-1, random_state=42
))
rf.fit(X_train, Y_train)

# ===================== TESTE =====================
X_test = np.hstack([X_te, T_te[:,None]])
Y_pred = rf.predict(X_test)

# ===================== MÉTRICAS =====================
def metrics_block(y_true, y_pred, title=""):
    y1, y2 = y_true.reshape(-1), y_pred.reshape(-1)
    R2   = r2_score(y1, y2)
    RMSE = np.sqrt(mean_squared_error(y1, y2))
    MAE  = mean_absolute_error(y1, y2)
    print(f"\n== {title} ==\nR2={R2:.4f} | RMSE={RMSE:.4f} | MAE={MAE:.4f}")
    return dict(R2=R2, RMSE=RMSE, MAE=MAE)

Y_ref_te = np.tile(y_ref, (X_te.shape[0],1))
metrics_block(Y_ref_te, Y_pred, "Predito vs Referência (20 °C)")
metrics_block(X_te,     Y_pred, "Predito vs Original")

# ===================== PLOT =====================
def plot_comp(i=0, save=False, prefix="rf_direct_curve"):
    plt.figure(figsize=(10,5))
    plt.plot(fkHz, X_te[i],     label=f"Original @ {T_te[i]:.0f} °C")
    plt.plot(fkHz, y_ref,       label=f"Referência @ {REF_TEMP} °C")
    plt.plot(fkHz, Y_pred[i],   label="Predito (compensado)")
    plt.xlabel("Frequência (kHz)")
    plt.ylabel("Re{Z}")
    plt.title(f"Amostra {i} — Compensação via RF direto")
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
    if save: plt.savefig(f"{prefix}_i{i}.png", dpi=300)
    plt.show()

# Exemplo
plot_comp(i=0)
